# Лабораторная работа 4

Студент: Захаров Никита Константинович
Группа: 6403-010302D


Tensorflow 2.x

1) Подготовка данных

2) Использование Keras Model API

3) Использование Keras Sequential + Functional API

https://www.tensorflow.org/tutorials

Для выполнения лабораторной работы необходимо установить tensorflow версии 2.0 или выше .

Рекомендуется использовать возможности Colab'а по обучению моделей на GPU.



In [1]:
import os
import tensorflow as tf
import numpy as np
import math
import timeit
import matplotlib.pyplot as plt

gpus = tf.config.list_physical_devices('GPU')
device = '/GPU:0' if gpus else '/CPU:0'

%matplotlib inline


I0000 00:00:1777552282.392227   28186 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777552282.532915   28186 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777552285.466905   28186 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
W0000 00:00:1777552288.573327   28186 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries men

# Подготовка данных
Загрузите набор данных из предыдущей лабораторной работы. 

In [2]:
def load_cifar10(num_training=49000, num_validation=1000, num_test=10000):
    """
    Fetch the CIFAR-10 dataset from the web and perform preprocessing to prepare
    it for the two-layer neural net classifier. These are the same steps as
    we used for the SVM, but condensed to a single function.
    """
    # Load the raw CIFAR-10 dataset and use appropriate data types and shapes
    cifar10 = tf.keras.datasets.cifar10.load_data()
    (X_train, y_train), (X_test, y_test) = cifar10
    X_train = np.asarray(X_train, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.int32).flatten()
    X_test = np.asarray(X_test, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.int32).flatten()

    # Subsample the data
    mask = range(num_training, num_training + num_validation)
    X_val = X_train[mask]
    y_val = y_train[mask]
    mask = range(num_training)
    X_train = X_train[mask]
    y_train = y_train[mask]
    mask = range(num_test)
    X_test = X_test[mask]
    y_test = y_test[mask]

    # Normalize the data: subtract the mean pixel and divide by std
    mean_pixel = X_train.mean(axis=(0, 1, 2), keepdims=True)
    std_pixel = X_train.std(axis=(0, 1, 2), keepdims=True)
    X_train = (X_train - mean_pixel) / std_pixel
    X_val = (X_val - mean_pixel) / std_pixel
    X_test = (X_test - mean_pixel) / std_pixel

    return X_train, y_train, X_val, y_val, X_test, y_test

# If there are errors with SSL downloading involving self-signed certificates,
# it may be that your Python version was recently installed on the current machine.
# See: https://github.com/tensorflow/tensorflow/issues/10779
# To fix, run the command: /Applications/Python\ 3.7/Install\ Certificates.command
#   ...replacing paths as necessary.

# Invoke the above function to get our data.
NHW = (0, 1, 2)
X_train, y_train, X_val, y_val, X_test, y_test = load_cifar10()
print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape, y_train.dtype)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

/home/ZakharovNK/.local/lib/python3.12/site-packages/keras/src/datasets/cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


Train data shape:  (49000, 32, 32, 3)
Train labels shape:  (49000,) int32
Validation data shape:  (1000, 32, 32, 3)
Validation labels shape:  (1000,)
Test data shape:  (10000, 32, 32, 3)
Test labels shape:  (10000,)


In [3]:
class Dataset(object):
    def __init__(self, X, y, batch_size, shuffle=False):
        """
        Construct a Dataset object to iterate over data X and labels y
        
        Inputs:
        - X: Numpy array of data, of any shape
        - y: Numpy array of labels, of any shape but with y.shape[0] == X.shape[0]
        - batch_size: Integer giving number of elements per minibatch
        - shuffle: (optional) Boolean, whether to shuffle the data on each epoch
        """
        assert X.shape[0] == y.shape[0], 'Got different numbers of data and labels'
        self.X, self.y = X, y
        self.batch_size, self.shuffle = batch_size, shuffle

    def __iter__(self):
        N, B = self.X.shape[0], self.batch_size
        idxs = np.arange(N)
        if self.shuffle:
            np.random.shuffle(idxs)
        return iter((self.X[i:i+B], self.y[i:i+B]) for i in range(0, N, B))


train_dset = Dataset(X_train, y_train, batch_size=64, shuffle=True)
val_dset = Dataset(X_val, y_val, batch_size=64, shuffle=False)
test_dset = Dataset(X_test, y_test, batch_size=64)

In [4]:
# We can iterate through a dataset like this:
for t, (x, y) in enumerate(train_dset):
    print(t, x.shape, y.shape)
    if t > 5: break

0 (64, 32, 32, 3) (64,)
1 (64, 32, 32, 3) (64,)
2 (64, 32, 32, 3) (64,)
3 (64, 32, 32, 3) (64,)
4 (64, 32, 32, 3) (64,)
5 (64, 32, 32, 3) (64,)
6 (64, 32, 32, 3) (64,)


#  Keras Model Subclassing API


Для реализации собственной модели с помощью Keras Model Subclassing API необходимо выполнить следующие шаги:

1) Определить новый класс, который является наследником tf.keras.Model.

2) В методе __init__() определить все необходимые слои из модуля tf.keras.layer

3) Реализовать прямой проход в методе call() на основе слоев, объявленных в __init__()

Ниже приведен пример использования keras API для определения двухслойной полносвязной сети. 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras

In [5]:
class TwoLayerFC(tf.keras.Model):
    def __init__(self, hidden_size, num_classes):
        super(TwoLayerFC, self).__init__()        
        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.fc1 = tf.keras.layers.Dense(hidden_size, activation='relu',
                                   kernel_initializer=initializer)
        self.fc2 = tf.keras.layers.Dense(num_classes, activation='softmax',
                                   kernel_initializer=initializer)
        self.flatten = tf.keras.layers.Flatten()
    
    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        return x


def test_TwoLayerFC():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    x = tf.zeros((64, input_size))
    model = TwoLayerFC(hidden_size, num_classes)
    with tf.device(device):
        scores = model(x)
        print(scores.shape)
        
test_TwoLayerFC()

(64, 10)


Реализуйте трехслойную CNN для вашей задачи классификации. 

Архитектура сети:
    
1. Сверточный слой (5 x 5 kernels, zero-padding = 'same')
2. Функция активации ReLU 
3. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
4. Функция активации ReLU 
5. Полносвязный слой 
6. Функция активации Softmax 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Conv2D

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dense

In [6]:
class ThreeLayerConvNet(tf.keras.Model):
    def __init__(self, channel_1, channel_2, num_classes):
        super(ThreeLayerConvNet, self).__init__()
        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.conv1 = tf.keras.layers.Conv2D(channel_1, 5, padding='same', activation='relu', kernel_initializer=initializer)
        self.conv2 = tf.keras.layers.Conv2D(channel_2, 3, padding='same', activation='relu', kernel_initializer=initializer)
        self.flatten = tf.keras.layers.Flatten()
        self.fc = tf.keras.layers.Dense(num_classes, activation='softmax', kernel_initializer=initializer)
        
    def call(self, x, training=False):
        if x.shape.rank == 4 and x.shape[1] == 3 and x.shape[-1] != 3:
            x = tf.transpose(x, [0, 2, 3, 1])
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.flatten(x)
        scores = self.fc(x)
        return scores


In [7]:
def test_ThreeLayerConvNet():    
    channel_1, channel_2, num_classes = 12, 8, 10
    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)
    with tf.device(device):
        x = tf.zeros((64, 3, 32, 32))
        scores = model(x)
        print(scores.shape)

test_ThreeLayerConvNet()

(64, 10)


Пример реализации процесса обучения:

In [10]:
print_every = 100

def train_part34(model_init_fn, optimizer_init_fn, num_epochs=1, is_training=False):
    with tf.device(device):
        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
        model = model_init_fn()
        optimizer = optimizer_init_fn()
        train_loss = tf.keras.metrics.Mean(name='train_loss')
        train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')
        val_loss = tf.keras.metrics.Mean(name='val_loss')
        val_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')
        steps_per_epoch = (train_dset.X.shape[0] + train_dset.batch_size - 1) // train_dset.batch_size
        total_steps = steps_per_epoch * num_epochs
        t = 0
        for epoch in range(num_epochs):
            train_loss.reset_state()
            train_accuracy.reset_state()
            for x_np, y_np in train_dset:
                with tf.GradientTape() as tape:
                    scores = model(x_np, training=is_training)
                    loss = loss_fn(y_np, scores)
                gradients = tape.gradient(loss, model.trainable_variables)
                optimizer.apply_gradients(zip(gradients, model.trainable_variables))
                train_loss.update_state(loss)
                train_accuracy.update_state(y_np, scores)
                if t % print_every == 0 or t == total_steps - 1:
                    val_loss.reset_state()
                    val_accuracy.reset_state()
                    for test_x, test_y in val_dset:
                        prediction = model(test_x, training=False)
                        t_loss = loss_fn(test_y, prediction)
                        val_loss.update_state(t_loss)
                        val_accuracy.update_state(test_y, prediction)
                    print(
                        f'step {t + 1}/{total_steps} | epoch {epoch + 1}/{num_epochs} | '
                        f'train_loss {train_loss.result().numpy():.4f} | '
                        f'train_acc {train_accuracy.result().numpy() * 100:.2f}% | '
                        f'val_loss {val_loss.result().numpy():.4f} | '
                        f'val_acc {val_accuracy.result().numpy() * 100:.2f}%'
                    )
                t += 1


In [11]:
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return TwoLayerFC(hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

0 1 tf.Tensor(2.8746014, shape=(), dtype=float32) tf.Tensor(0.09375, shape=(), dtype=float32) tf.Tensor(2.8666167, shape=(), dtype=float32) tf.Tensor(0.15, shape=(), dtype=float32)
100 1 tf.Tensor(2.2386086, shape=(), dtype=float32) tf.Tensor(0.28279704, shape=(), dtype=float32) tf.Tensor(1.9108676, shape=(), dtype=float32) tf.Tensor(0.368, shape=(), dtype=float32)
200 1 tf.Tensor(2.0813644, shape=(), dtype=float32) tf.Tensor(0.31941852, shape=(), dtype=float32) tf.Tensor(1.8489586, shape=(), dtype=float32) tf.Tensor(0.398, shape=(), dtype=float32)
300 1 tf.Tensor(2.0025067, shape=(), dtype=float32) tf.Tensor(0.3381956, shape=(), dtype=float32) tf.Tensor(1.896107, shape=(), dtype=float32) tf.Tensor(0.375, shape=(), dtype=float32)
400 1 tf.Tensor(1.9332572, shape=(), dtype=float32) tf.Tensor(0.35699812, shape=(), dtype=float32) tf.Tensor(1.735019, shape=(), dtype=float32) tf.Tensor(0.416, shape=(), dtype=float32)
500 1 tf.Tensor(1.8912495, shape=(), dtype=float32) tf.Tensor(0.36729667, 

Обучите трехслойную CNN. В tf.keras.optimizers.SGD укажите Nesterov momentum = 0.9 . 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/optimizers/SGD

Значение accuracy на валидационной выборке после 1 эпохи обучения должно быть > 50% .

In [12]:
learning_rate = 3e-3
channel_1, channel_2, num_classes = 32, 16, 10

def model_init_fn():
    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)
    return model

def optimizer_init_fn():
    optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9, nesterov=True)
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)


0 1 tf.Tensor(3.4054985, shape=(), dtype=float32) tf.Tensor(0.171875, shape=(), dtype=float32) tf.Tensor(4.3314705, shape=(), dtype=float32) tf.Tensor(0.099, shape=(), dtype=float32)
100 1 tf.Tensor(1.9032468, shape=(), dtype=float32) tf.Tensor(0.3387995, shape=(), dtype=float32) tf.Tensor(1.5965309, shape=(), dtype=float32) tf.Tensor(0.46, shape=(), dtype=float32)
200 1 tf.Tensor(1.7390429, shape=(), dtype=float32) tf.Tensor(0.39171332, shape=(), dtype=float32) tf.Tensor(1.4604023, shape=(), dtype=float32) tf.Tensor(0.503, shape=(), dtype=float32)
300 1 tf.Tensor(1.6491481, shape=(), dtype=float32) tf.Tensor(0.41855273, shape=(), dtype=float32) tf.Tensor(1.3946503, shape=(), dtype=float32) tf.Tensor(0.506, shape=(), dtype=float32)
400 1 tf.Tensor(1.5759914, shape=(), dtype=float32) tf.Tensor(0.44365647, shape=(), dtype=float32) tf.Tensor(1.331221, shape=(), dtype=float32) tf.Tensor(0.516, shape=(), dtype=float32)
500 1 tf.Tensor(1.5249432, shape=(), dtype=float32) tf.Tensor(0.46129617

# Использование Keras Sequential API для реализации последовательных моделей.

Пример для полносвязной сети:

In [13]:
learning_rate = 1e-2

def model_init_fn():
    input_shape = (32, 32, 3)
    hidden_layer_size, num_classes = 4000, 10
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    layers = [
        tf.keras.layers.Flatten(input_shape=input_shape),
        tf.keras.layers.Dense(hidden_layer_size, activation='relu',
                              kernel_initializer=initializer),
        tf.keras.layers.Dense(num_classes, activation='softmax', 
                              kernel_initializer=initializer),
    ]
    model = tf.keras.Sequential(layers)
    return model

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate) 

train_part34(model_init_fn, optimizer_init_fn)

/home/ZakharovNK/.local/lib/python3.12/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


0 1 tf.Tensor(3.0035832, shape=(), dtype=float32) tf.Tensor(0.140625, shape=(), dtype=float32) tf.Tensor(2.8121774, shape=(), dtype=float32) tf.Tensor(0.126, shape=(), dtype=float32)
100 1 tf.Tensor(2.2540095, shape=(), dtype=float32) tf.Tensor(0.28480816, shape=(), dtype=float32) tf.Tensor(1.8883166, shape=(), dtype=float32) tf.Tensor(0.381, shape=(), dtype=float32)
200 1 tf.Tensor(2.0819502, shape=(), dtype=float32) tf.Tensor(0.3238495, shape=(), dtype=float32) tf.Tensor(1.8254094, shape=(), dtype=float32) tf.Tensor(0.401, shape=(), dtype=float32)
300 1 tf.Tensor(2.0030143, shape=(), dtype=float32) tf.Tensor(0.34141403, shape=(), dtype=float32) tf.Tensor(1.879442, shape=(), dtype=float32) tf.Tensor(0.366, shape=(), dtype=float32)
400 1 tf.Tensor(1.9318429, shape=(), dtype=float32) tf.Tensor(0.36066085, shape=(), dtype=float32) tf.Tensor(1.749284, shape=(), dtype=float32) tf.Tensor(0.417, shape=(), dtype=float32)
500 1 tf.Tensor(1.8877298, shape=(), dtype=float32) tf.Tensor(0.37125748

Альтернативный менее гибкий способ обучения:

In [ ]:
model = model_init_fn()
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

Перепишите реализацию трехслойной CNN с помощью tf.keras.Sequential API . Обучите модель двумя способами.

In [15]:
def model_init_fn():
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    model = tf.keras.Sequential([
        tf.keras.layers.InputLayer(input_shape=(32, 32, 3)),
        tf.keras.layers.Conv2D(32, 5, padding='same', activation='relu', kernel_initializer=initializer),
        tf.keras.layers.Conv2D(16, 3, padding='same', activation='relu', kernel_initializer=initializer),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(10, activation='softmax', kernel_initializer=initializer),
    ])
    return model

learning_rate = 5e-4
def optimizer_init_fn():
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)


/home/ZakharovNK/.local/lib/python3.12/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


0 1 tf.Tensor(3.106831, shape=(), dtype=float32) tf.Tensor(0.140625, shape=(), dtype=float32) tf.Tensor(4.274853, shape=(), dtype=float32) tf.Tensor(0.087, shape=(), dtype=float32)
100 1 tf.Tensor(1.9997306, shape=(), dtype=float32) tf.Tensor(0.31064355, shape=(), dtype=float32) tf.Tensor(1.6624423, shape=(), dtype=float32) tf.Tensor(0.432, shape=(), dtype=float32)
200 1 tf.Tensor(1.806878, shape=(), dtype=float32) tf.Tensor(0.3722015, shape=(), dtype=float32) tf.Tensor(1.5211489, shape=(), dtype=float32) tf.Tensor(0.465, shape=(), dtype=float32)
300 1 tf.Tensor(1.7094548, shape=(), dtype=float32) tf.Tensor(0.40448505, shape=(), dtype=float32) tf.Tensor(1.4518684, shape=(), dtype=float32) tf.Tensor(0.495, shape=(), dtype=float32)
400 1 tf.Tensor(1.6363559, shape=(), dtype=float32) tf.Tensor(0.42900562, shape=(), dtype=float32) tf.Tensor(1.4052687, shape=(), dtype=float32) tf.Tensor(0.505, shape=(), dtype=float32)
500 1 tf.Tensor(1.5813681, shape=(), dtype=float32) tf.Tensor(0.44726172,

In [16]:
model = model_init_fn()
model.compile(optimizer='sgd',
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

766/766 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - loss: 1.5914 - sparse_categorical_accuracy: 0.4421 - val_loss: 1.3724 - val_sparse_categorical_accuracy: 0.5150
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 1.3750 - sparse_categorical_accuracy: 0.5121


[1.375032663345337, 0.5120999813079834]

# Использование Keras Functional API

Для реализации более сложных архитектур сети с несколькими входами/выходами, повторным использованием слоев, "остаточными" связями (residual connections) необходимо явно указать входные и выходные тензоры. 

Ниже представлен пример для полносвязной сети. 

In [17]:
def two_layer_fc_functional(input_shape, hidden_size, num_classes):  
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    inputs = tf.keras.Input(shape=input_shape)
    flattened_inputs = tf.keras.layers.Flatten()(inputs)
    fc1_output = tf.keras.layers.Dense(hidden_size, activation='relu',
                                 kernel_initializer=initializer)(flattened_inputs)
    scores = tf.keras.layers.Dense(num_classes, activation='softmax',
                             kernel_initializer=initializer)(fc1_output)

    # Instantiate the model given inputs and outputs.
    model = tf.keras.Model(inputs=inputs, outputs=scores)
    return model

def test_two_layer_fc_functional():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    input_shape = (50,)
    
    x = tf.zeros((64, input_size))
    model = two_layer_fc_functional(input_shape, hidden_size, num_classes)
    
    with tf.device(device):
        scores = model(x)
        print(scores.shape)
        
test_two_layer_fc_functional()

(64, 10)


In [18]:
input_shape = (32, 32, 3)
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return two_layer_fc_functional(input_shape, hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

0 1 tf.Tensor(3.5956511, shape=(), dtype=float32) tf.Tensor(0.03125, shape=(), dtype=float32) tf.Tensor(2.9410787, shape=(), dtype=float32) tf.Tensor(0.104, shape=(), dtype=float32)
100 1 tf.Tensor(2.2343056, shape=(), dtype=float32) tf.Tensor(0.27908415, shape=(), dtype=float32) tf.Tensor(1.9104542, shape=(), dtype=float32) tf.Tensor(0.376, shape=(), dtype=float32)
200 1 tf.Tensor(2.0808587, shape=(), dtype=float32) tf.Tensor(0.3146766, shape=(), dtype=float32) tf.Tensor(1.8878422, shape=(), dtype=float32) tf.Tensor(0.371, shape=(), dtype=float32)
300 1 tf.Tensor(2.0011203, shape=(), dtype=float32) tf.Tensor(0.33502907, shape=(), dtype=float32) tf.Tensor(1.9346999, shape=(), dtype=float32) tf.Tensor(0.369, shape=(), dtype=float32)
400 1 tf.Tensor(1.9323753, shape=(), dtype=float32) tf.Tensor(0.3539199, shape=(), dtype=float32) tf.Tensor(1.7651306, shape=(), dtype=float32) tf.Tensor(0.403, shape=(), dtype=float32)
500 1 tf.Tensor(1.8886145, shape=(), dtype=float32) tf.Tensor(0.36495757

Поэкспериментируйте с архитектурой сверточной сети. Для вашего набора данных вам необходимо получить как минимум 70% accuracy на валидационной выборке за 10 эпох обучения. Опишите все эксперименты и сделайте выводы (без выполнения данного пункта работы приниматься не будут). 

Эспериментируйте с архитектурой, гиперпараметрами, функцией потерь, регуляризацией, методом оптимизации.  

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/BatchNormalization#methods https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dropout#methods

In [19]:
class CustomConvNet(tf.keras.Model):
    def __init__(self):
        super(CustomConvNet, self).__init__()
        initializer = tf.initializers.HeNormal()
        self.conv1 = tf.keras.layers.Conv2D(64, 3, padding='same', use_bias=False, kernel_initializer=initializer)
        self.bn1 = tf.keras.layers.BatchNormalization()
        self.conv2 = tf.keras.layers.Conv2D(64, 3, padding='same', use_bias=False, kernel_initializer=initializer)
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.pool1 = tf.keras.layers.MaxPool2D()
        self.drop1 = tf.keras.layers.Dropout(0.25)
        self.conv3 = tf.keras.layers.Conv2D(128, 3, padding='same', use_bias=False, kernel_initializer=initializer)
        self.bn3 = tf.keras.layers.BatchNormalization()
        self.conv4 = tf.keras.layers.Conv2D(128, 3, padding='same', use_bias=False, kernel_initializer=initializer)
        self.bn4 = tf.keras.layers.BatchNormalization()
        self.pool2 = tf.keras.layers.MaxPool2D()
        self.drop2 = tf.keras.layers.Dropout(0.35)
        self.conv5 = tf.keras.layers.Conv2D(256, 3, padding='same', use_bias=False, kernel_initializer=initializer)
        self.bn5 = tf.keras.layers.BatchNormalization()
        self.gap = tf.keras.layers.GlobalAveragePooling2D()
        self.fc1 = tf.keras.layers.Dense(256, activation='relu', kernel_initializer=initializer)
        self.drop3 = tf.keras.layers.Dropout(0.5)
        self.fc2 = tf.keras.layers.Dense(10, activation='softmax', kernel_initializer=initializer)
    
    def call(self, input_tensor, training=False):
        x = input_tensor
        x = self.conv1(x)
        x = self.bn1(x, training=training)
        x = tf.nn.relu(x)
        x = self.conv2(x)
        x = self.bn2(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool1(x)
        x = self.drop1(x, training=training)
        x = self.conv3(x)
        x = self.bn3(x, training=training)
        x = tf.nn.relu(x)
        x = self.conv4(x)
        x = self.bn4(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool2(x)
        x = self.drop2(x, training=training)
        x = self.conv5(x)
        x = self.bn5(x, training=training)
        x = tf.nn.relu(x)
        x = self.gap(x)
        x = self.fc1(x)
        x = self.drop3(x, training=training)
        x = self.fc2(x)
        return x


print_every = 700
num_epochs = 10

model = CustomConvNet()

def model_init_fn():
    return CustomConvNet()

def optimizer_init_fn():
    learning_rate = 1e-3
    return tf.keras.optimizers.Adam(learning_rate) 

train_part34(model_init_fn, optimizer_init_fn, num_epochs=num_epochs, is_training=True)


0 1 tf.Tensor(2.597557, shape=(), dtype=float32) tf.Tensor(0.0625, shape=(), dtype=float32) tf.Tensor(4.3806286, shape=(), dtype=float32) tf.Tensor(0.135, shape=(), dtype=float32)
700 1 tf.Tensor(1.450685, shape=(), dtype=float32) tf.Tensor(0.46990907, shape=(), dtype=float32) tf.Tensor(1.250111, shape=(), dtype=float32) tf.Tensor(0.548, shape=(), dtype=float32)
1400 2 tf.Tensor(1.0841663, shape=(), dtype=float32) tf.Tensor(0.61151576, shape=(), dtype=float32) tf.Tensor(1.0671325, shape=(), dtype=float32) tf.Tensor(0.637, shape=(), dtype=float32)
2100 3 tf.Tensor(0.93303186, shape=(), dtype=float32) tf.Tensor(0.66808546, shape=(), dtype=float32) tf.Tensor(0.86242646, shape=(), dtype=float32) tf.Tensor(0.703, shape=(), dtype=float32)
2800 4 tf.Tensor(0.828913, shape=(), dtype=float32) tf.Tensor(0.7086233, shape=(), dtype=float32) tf.Tensor(1.3003999, shape=(), dtype=float32) tf.Tensor(0.614, shape=(), dtype=float32)
3500 5 tf.Tensor(0.7516608, shape=(), dtype=float32) tf.Tensor(0.738808

Опишите все эксперименты, результаты. Сделайте выводы.

Сначала была обучена двухслойная полносвязная сеть через Model Subclassing API: validation accuracy выросла примерно до 0.44. Затем были проверены эквивалентные реализации через Sequential и Functional API, которые дали очень близкий результат на той же архитектуре. Базовая трёхслойная CNN оказалась заметно сильнее полносвязных моделей и подняла validation accuracy примерно до 0.56. После этого архитектура была усложнена: добавлены дополнительные сверточные блоки, BatchNormalization, MaxPooling, Dropout и GlobalAveragePooling. Лучшая модель с блоками 64-64-128-128-256 и оптимизатором Adam с learning_rate=1e-3 превысила требуемый порог 70% уже к 3-й эпохе и завершила обучение с validation accuracy около 0.775 на 10-й эпохе. Вывод: для CIFAR-10 основной прирост качества дали увеличение глубины сверточной сети, нормализация и регуляризация, а требование задания по достижению не менее 70% accuracy на валидации выполнено.
